# Feature Engineering

- Transform preprocessed data from `cookie`, `index`, `method`, `url`, `host`, and `label`, `payload` into numerical or categorical features that can be used for modeling.
- Focus on features that distinguish anomalous requests (anom) from normal requests (norm).
- Address missing values in contentLength (e.g., fill with 0 or add a binary flag).
- Convert categorical columns `method`, `label`, `host` into a format suitable for machine learning (e.g., one-hot encoding, label encoding).
- Extract features that capture HTTP request characteristics (e.g., SQL injection patterns, payload length, URL length).
- Save the transformed DataFrame with features as `csic_2010_featurised.csv` for modeling.

In [1]:
## 1. Load the Preprocessed Data

In [2]:
import os
import re
import random
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import OneHotEncoder

path = Path.cwd()
data_path = path.parent / "data"
input_path = data_path / "csic_2010_preprocessed.csv" 
output_path = data_path / "csic_2010_featurised.csv" 
# Load the DataFrame with keep_default_na=False to preserve empty strings
#df = pd.read_csv(input_path, keep_default_na=False)

In [3]:
# Load the Parquet file
%time
df = pd.read_parquet(input_path.with_suffix('.parquet'))

CPU times: user 11 μs, sys: 1e+03 ns, total: 12 μs
Wall time: 28.6 μs


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   cookie         61065 non-null  object 
 1   index          61065 non-null  int64  
 2   method         61065 non-null  object 
 3   url            61065 non-null  object 
 4   host           61065 non-null  object 
 5   label          61065 non-null  object 
 6   payload        61065 non-null  object 
 7   contentLength  17977 non-null  float64
dtypes: float64(1), int64(1), object(6)
memory usage: 3.7+ MB


## 2. Handle Missing Values

- Handle missing `contentLength` - fill with `0`
- Handle Empty `payload` - add binary flag to indicate whether re has payload

In [5]:
df['contentLength'] = df['contentLength'].fillna(0)

## 3. Analyse column values to derive features

In [6]:
print(f"cookie:        {df['cookie'].nunique()}")
print(f"index:         {df['index'].nunique()}")
print(f"method:        {df['method'].nunique()}")
print(f"url:           {df['url'].nunique()}")
print(f"host:          {df['host'].nunique()}")
print(f"label:         {df['label'].nunique()}")
print(f"payload:       {df['payload'].nunique()}")
print(f"contentLength: {df['contentLength'].nunique()}")

cookie:        61065
index:         36000
method:        3
url:           1643
host:          2
label:         2
payload:       12093
contentLength: 383


`index` column is non-unique and reused across requests so it is not useful in modeling. Drop the column.

In [7]:
df_featurised = df.drop(columns = 'index')
#df_featurised.drop(columns=["method"], inplace=True)
df_featurised.columns.tolist()

['cookie', 'method', 'url', 'host', 'label', 'payload', 'contentLength']

### 3.1 Analyse `cookie` column values

In [8]:
# Random cookie samples
random_row_numbers = random.sample(range(0, 61066), 5)
for rn in random_row_numbers:
    print(f"row#{rn}: {df_featurised.iloc[rn]['cookie']}")

row#10034: JSESSIONID=296B91DAD7801452F4FA4CAE43F065DF
row#24408: JSESSIONID=6556B971B35EF66094D213C0A3C80B7D
row#60487: JSESSIONID=FD76975539EBC901C0B7E5C546F8BEC0
row#7875: JSESSIONID=209F543507EB64DF6D930E6C5B2BE296
row#20557: JSESSIONID=556CC0DBBEC21E58594C87D9F48A8C8E


In [9]:
# Compute the length of each cookie
cookie_lengths = df_featurised['cookie'].str.len()

print(f"Min len: {cookie_lengths.min()}")
print(f"Max len: {cookie_lengths.max()}")
print(f"mean len: {cookie_lengths.mean():.2f}")
#print(f"Unique lengths: {cookie_lengths.nunique()}")

# Count how many cookies match the pattern
cookie_pattern=r'^JSESSIONID=[A-Z0-9]+$'
valid_count = df_featurised['cookie'].str.match(cookie_pattern).sum()
print(f"Number of valid cookies: {valid_count}")
#print(f"Total cookies: {len(df_featurised)}")

Min len: 43
Max len: 43
mean len: 43.00
Number of valid cookies: 61065


All cookies have a length of `43` characters and a consistent format. This implies that the cookie column does not provide any variability that could distinguish between normal (norm) and anomalous (anom) requests. Drop the `cookie` Column from the Feature set

In [10]:
df_featurised = df_featurised.drop(columns = 'cookie')
df_featurised.columns.tolist()

['method', 'url', 'host', 'label', 'payload', 'contentLength']

### 3.2 Analyse `Method` column values

In [11]:
df_featurised['method'].value_counts()

method
GET     43088
POST    17580
PUT       397
Name: count, dtype: int64

Indicates type of HTTP requests - majority are `GET` `43,088` followed by `POST` `17,580` and the `PUT` `397` which can influence anomaly detection  
e.g., `POST` and `PUT` requests are more likely to contain payloads or be anomalous  

Use one-hot encoding to transform method categories into numerical features.

In [12]:
# Initialize the encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
category_col = ["method"]

# Fit the encoder to the data
encoder.fit(df_featurised[category_col])
print(f"Encoder categories:\n\t{encoder.categories_}")

# Generate column names for the encoded features
encoded_cols = list(encoder.get_feature_names_out(category_col))
print(f"Generated column names:\n\t{encoded_cols}")

Encoder categories:
	[array(['GET', 'POST', 'PUT'], dtype=object)]
Generated column names:
	['method_GET', 'method_POST', 'method_PUT']


The method column is transformed into three binary features - `method_GET`, `method_POST`, `method_PUT`. This allows the model to interpret the request type as numerical input while preserving the categorical relationships.

Apply the encoder to the dataset and Concatenate the encoded features with the original DataFrame for modeling.

In [13]:
df_featurised[encoded_cols] = encoder.transform(df_featurised[category_col]).copy()

In [14]:
# Drop the original 'method' column as it is longer needed
df_featurised.drop(columns=["method"], inplace=True)

In [15]:
print(df_featurised.columns.tolist())

['url', 'host', 'label', 'payload', 'contentLength', 'method_GET', 'method_POST', 'method_PUT']


### 3.3 Analyse `host` column values

In [17]:
df_featurised['host'].value_counts()

host
localhost:8080    60668
localhost:9090      397
Name: count, dtype: int64

**Observations:**  
- Identifies the `server:port` for the request.
- Contains two host values `localhost:8080` and `localhost:9090`  
- Majority are `localhost:8080` `60,668` following by `localhost:9090` `397`
- Useful for filtering or grouping requests by host.
- `host` column is highly imbalanced, which may limit its predictive power.

In [18]:
# Initialize the encoder
host_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
category_col = ["host"]

# Fit the encoder to the data
host_encoder.fit(df_featurised[category_col])
print(f"Encoder categories:\n\t{host_encoder.categories_}")

# Generate column names for the encoded features
host_encoded_cols = list(host_encoder.get_feature_names_out(category_col))
print(f"Generated column names:\n\t{host_encoded_cols}")

Encoder categories:
	[array(['localhost:8080', 'localhost:9090'], dtype=object)]
Generated column names:
	['host_localhost:8080', 'host_localhost:9090']


The host column is transformed into two binary features - method_GET, method_POST, method_PUT. This allows the model to interpret the request type as numerical input while preserving the categorical relationships.

Apply the encoder to the dataset and concatenate the encoded features with the original DataFrame for modeling.

In [19]:
df_featurised[host_encoded_cols] = host_encoder.transform(df_featurised[category_col]).copy()
df_featurised.head(5)

,url,host,label,payload,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,host_localhost:9090
0,http://localhost:8080/tienda1/imagenes/2.gif,localhost:8080,norm,,0.0,1.0,0.0,0.0,1.0,0.0
1,http://localhost:8080/tienda1/publico/vaciar.j...,localhost:8080,anom,,0.0,1.0,0.0,0.0,1.0,0.0
2,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,norm,id=2,0.0,1.0,0.0,0.0,1.0,0.0
3,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,anom,id=3 nombre=Jam%F3n+Ib%E9rico precio=%2500 can...,0.0,1.0,0.0,0.0,1.0,0.0
4,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,anom,id=2sessionid%3D12312312%26+username%3D%3Cscri...,174.0,0.0,1.0,0.0,1.0,0.0


### 3.4 Analyse URL column values

Count unique URLs in the dataset

In [20]:
print(f"Unique URLs: {df_featurised['url'].nunique()}")

Unique URLs: 1643


Sample a few random URLs for analysis

In [21]:
random_row_numbers = random.sample(range(0, 61066), 10)
for rn in random_row_numbers:
    print(f"row[{rn}]: {df_featurised.iloc[rn]['url']}")

row[39214]: http://localhost:8080/tienda1/publico/anadir.jsp
row[32101]: http://localhost:8080/tienda1/publico/pagar.jsp
row[1869]: http://localhost:8080/tienda1/publico/autenticar.jsp
row[20269]: http://localhost:8080/tienda1/global/estilos.css
row[29143]: http://localhost:8080/tienda1/publico/anadir.jsp
row[36847]: http://localhost:8080/tienda1/publico/carrito.jsp
row[26852]: http://localhost:8080/tienda1/publico/vaciar.jsp
row[16528]: http://localhost:8080/tienda1/imagenes/cmenbul.gif
row[3357]: http://localhost:8080/tienda1/publico/autenticar.jsp
row[17299]: http://localhost:8080/tienda1/imagenes/2.gif


**Observations:**  
URLs contain patterns, structures, and metadata that can be leveraged for feature engineering.  
Eg:
- url length
- Count of special chars: `/` `.` `?` `&` etc
- Presence of query prarams
- Presence of suspecious patterns: sql select, union, drop etc
- Presence of path traversal

Analyse `urls` in detail for presence of `special chars`, `patterns` etc that are `label`'ed  anomalous

In [22]:
from collections import defaultdict

#url_df = df_featurised["url"].head(10).to_frame()
#url_df = df_featurised["url", "label"].to_frame()
url_df = df_featurised[["url", "label"]]

# Special characters
special_chars = ['/', '.', '?', '&', '=', '%']
total_urls_per_special_char = defaultdict(int)

# SQL patterns
sql_patterns = [ "select", "union", "drop", "insert", "delete"]
total_urls_per_sql_pattern = defaultdict(int)
for pattern in sql_patterns:
    total_urls_per_sql_pattern[pattern] = 0

# Script Pattern
script_regex_pattern = r"<script|%3cscript"
total_urls_with_script_pattern = 0

# Alert Pattern
alert_regex_pattern = r"alert|%61%6c%65%72%74"
total_urls_with_alert_pattern = 0

# OR pattern
or_regex_pattern = r"\bor\b|\|\|"
total_urls_with_or_pattern = 0

# Comment pattern
comment_regex_pattern = r"--|/\*|\*/|%2d%2d"
total_urls_with_comment_pattern = 0

# Path traversal pattern
path_traversal_regex_pattern = r"\.\./|%2e%2e%2f"
total_urls_with_path_traversal_pattern = 0

# Parameter pattern
parameter_regex_pattern = r"[?&][^=&]+="
total_urls_with_parameter_pattern = 0

# Trailing digits
trailing_digits_regex_pattern= r"http://localhost:8080[^0-9]*(\d+)"
total_urls_with_trailing_digits_pattern = 0

# Numeric ID
numeric_id_regex_pattern = r"/\d{10,}"
total_urls_with_numeric_id_pattern = 0

# Track label distribution for each pattern
label_distribution = {
    "select": {"anom": 0, "norm": 0},
    "drop": {"anom": 0, "norm": 0},
    "script": {"anom": 0, "norm": 0},
    "alert": {"anom": 0, "norm": 0},
    "or": {"anom": 0, "norm": 0},
    "comment": {"anom": 0, "norm": 0},
    "path_traversal": {"anom": 0, "norm": 0},
    "parameter": {"anom": 0, "norm": 0},
    "trailing_digits": {"anom": 0, "norm": 0},
    "numeric_id": {"anom": 0, "norm": 0},
}

# Iterate through each URL and its corresponding label
for url, label in zip(url_df["url"], url_df["label"]):

    # Count special characters
    for char in special_chars:
        count = url.count(char)
        if count > 0:
            total_urls_per_special_char[char] += 1
        else:
            total_urls_per_special_char[char] += 0

    for pattern in sql_patterns:
        if pattern in url:
          total_urls_per_sql_pattern[pattern] +=  1
          label_distribution[pattern][label] += 1
        
    if re.search(script_regex_pattern, url):
        total_urls_with_script_pattern += 1
        label_distribution["script"][label] += 1
        
    if re.search(alert_regex_pattern, url):
        total_urls_with_alert_pattern += 1
        label_distribution["alert"][label] += 1

    if re.search(or_regex_pattern, url):
        total_urls_with_or_pattern += 1
        label_distribution["or"][label] += 1
        
    if re.search(comment_regex_pattern, url):
        total_urls_with_comment_pattern += 1
        label_distribution["comment"][label] += 1
        
    if re.search(path_traversal_regex_pattern, url):
        total_urls_with_path_traversal_pattern += 1
        label_distribution["path_traversal"][label] += 1

    if re.search(parameter_regex_pattern, url):
        total_urls_with_parameter_pattern += 1
        label_distribution["parameter"][label] += 1
    
    if re.search(trailing_digits_regex_pattern, url):
        total_urls_with_trailing_digits_pattern += 1
        label_distribution["trailing_digits"][label] += 1
        
    if re.search(numeric_id_regex_pattern, url):
        total_urls_with_numeric_id_pattern += 1
        label_distribution["numeric_id"][label] += 1
    
# Display the total count of URLs containing each special character
print("Total URLs containing per special Character:")
for char, count in total_urls_per_special_char.items():
    print(f"    {char}: {count} URLs")

print("Total URLs containing per SQL pattern:")
for pattern, count in total_urls_per_sql_pattern.items():
    print(f"    {pattern}: {count} URLs")

print(f"Total URLs containing other patterns:")
print(f"   'or': {total_urls_with_or_pattern}")
print(f"   'script': {total_urls_with_script_pattern}")
print(f"   'alert': {total_urls_with_alert_pattern}")
print(f"   'comment': {total_urls_with_comment_pattern}")
print(f"   'path traversal': {total_urls_with_path_traversal_pattern}")
print(f"   'parameter': {total_urls_with_parameter_pattern}")
print(f"   'trailing digits': {total_urls_with_trailing_digits_pattern}")
print(f"   'numeric id': {total_urls_with_numeric_id_pattern}")

# Display label distribution for each pattern
print("\nLabel distribution for patterns:")
for pattern, counts in label_distribution.items():
    print(f"   {pattern}: {counts}")


Total URLs containing per special Character:
    /: 61065 URLs
    .: 60386 URLs
    ?: 0 URLs
    &: 0 URLs
    =: 0 URLs
    %: 0 URLs
Total URLs containing per SQL pattern:
    select: 8 URLs
    union: 0 URLs
    drop: 5 URLs
    insert: 0 URLs
    delete: 0 URLs
Total URLs containing other patterns:
   'or': 0
   'script': 0
   'alert': 0
   'comment': 0
   'path traversal': 0
   'parameter': 0
   'trailing digits': 59329
   'numeric id': 967

Label distribution for patterns:
   select: {'anom': 8, 'norm': 0}
   drop: {'anom': 5, 'norm': 0}
   script: {'anom': 0, 'norm': 0}
   alert: {'anom': 0, 'norm': 0}
   or: {'anom': 0, 'norm': 0}
   comment: {'anom': 0, 'norm': 0}
   path_traversal: {'anom': 0, 'norm': 0}
   parameter: {'anom': 0, 'norm': 0}
   trailing_digits: {'anom': 23329, 'norm': 36000}
   numeric_id: {'anom': 967, 'norm': 0}


**Observations:**

- Special Characters:  
  - `/` and `.` are too common and do not help distinguish anomalies.  
  - `?`, `&`, `=`, `%` do not appear in the dataset, so they provide no signal.  
  > **Conclusion:** Exclude all special character counts from the model

- SQL Patterns:  
  - `select` and `drop` are rare but meaningful for detecting SQL injection.  
  - `union`, `insert`, `delete` do not appear in the dataset.
  > **Conclusion:** Include `select` and `drop` as binary features to for detecting SQL injection attempts.

- Other Patterns:  
  - All patterns `or`, `script`, `alert`, `comment`, `path traversal`, `parameter` have zero occurrences in the dataset.  
  - `Trailing digits` appears in both anom and norm URLs and is not discriminative
  - `Numeric ID` is a strong indicator of anomalies. All of its occurrences are in anom URLs.
  > **Conclusion:** Include `numeric_id` as a binary feature. Exclude all other patterns from the model.

Add the `url` derived feature columns to dataframe

In [23]:
df_featurised["url_has_select"] = df_featurised["url"].str.contains("select", case=False).astype(int)
df_featurised["url_has_drop"] = df_featurised["url"].str.contains("drop", case=False).astype(int)
df_featurised["url_has_numeric_id"] = df_featurised["url"].apply(lambda x: 1 if re.search(r"/\d{10,}", x) else 0).astype(int)

# Additional features
df_featurised["url_length"] = df_featurised["url"].apply(len)

# Display the new features
#df_featurised[["url", "label", "url_has_select", "url_has_drop", "url_has_numeric_id", "url_length"]].head()
df_featurised.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   url                  61065 non-null  object 
 1   host                 61065 non-null  object 
 2   label                61065 non-null  object 
 3   payload              61065 non-null  object 
 4   contentLength        61065 non-null  float64
 5   method_GET           61065 non-null  float64
 6   method_POST          61065 non-null  float64
 7   method_PUT           61065 non-null  float64
 8   host_localhost:8080  61065 non-null  float64
 9   host_localhost:9090  61065 non-null  float64
 10  url_has_select       61065 non-null  int64  
 11  url_has_drop         61065 non-null  int64  
 12  url_has_numeric_id   61065 non-null  int64  
 13  url_length           61065 non-null  int64  
dtypes: float64(6), int64(4), object(4)
memory usage: 6.5+ MB


### 3.5 Analyse `payload` column values

Count unique `payloads` in the dataset

In [24]:
print(f"Unique Payloads: {df_featurised['payload'].nunique()}")

Unique Payloads: 12093


Sample a few random Payloads for analysis

In [25]:
unique_values = df_featurised['payload'].unique()
random_row_numbers = random.sample(range(0, len(unique_values)), 10)
for rn in random_row_numbers:
    print(f"    {unique_values[rn]}\n")

    modo=registro login=yaser password=jA3AL%CDn nombre=Neri apellidos=Crisp%ED+Vivo email=chopel%40restauranteslaespada%F1a.bg dni=21236992B direccion=San+Jorge%2C+188+ ciudad=Y%E9sero cp=10380 provincia=Badajoz ntc=9771406974624162 B1=Registrar

    id=3%2F nombre=Queso+Manchego precio=85 cantidad=45 B1=A%F1adir+al+carrito

    modo=registro login=hache6 password=den21o9a nombre=M%E1ximo apellidos=Sendr%F3s+Tudanca email=backy%40legami.tz dni=85855294C direccion=Glorieta+Farmaceutico+Perez+Jimenez+197%2C+ ciudad=L%FAcar cp=16741 provincia=Lugo ntc=5481930163785901 B1=Registrar

    modo=entrar login=bryant pwd=0NSu2isO remember=on B1=Entrar

    id=1 nombre=Queso+Manchego precio=85 cantidad=%253CSCRIPT%253Ealert%2528%2522Paros%2522%2529%253B%253C%252FSCRIPT%253E B1=A%F1adir+al+carrito

    modo=registro login=fleugel7 password=EmBAlDOsADa nombre=Itter apellidos=Ikediek+Torrades email=axman%40culturalsanpedrocf.ie dni=89590939M direccion=C%2F+Historiador+Manuel+Salcines+133%2C+6B ciud

**Observations:**  
- Payloads show form submissions, login attempts, data entries etc 
- Typically formatted as `key-value` pairs (e.g., modo=registro, login=quek, password=5a69io99t3).
- Feature can be derived based on structure, encoded chars, sql keywords, long numeric string, suspicious patterns etc

Analyse payloads in detail for presence of special chars, patterns etc that are label'ed anomalous

In [26]:
from collections import defaultdict

payload_df = df_featurised[["payload", "label"]]

# Special characters
special_chars = ['&', '=', '%']
total_payloads_per_special_char = defaultdict(int)

# Count digits
digits_regex_pattern= r"\d"
total_payloads_with_digits_pattern = 0

# Parameter pattern
#parameter_regex_pattern = r"[?&][^=&]+="
parameter_regex_pattern = r"(^|&)[^=&]+="
total_payloads_with_parameter_pattern = 0

# Quote pattern
quote_regex_pattern = r"'|%27|\"|%22"
total_payloads_with_quote_pattern = 0

# Angle bracket pattern
angle_bkt_regex_pattern = r"<|>|%3c|%3e"
total_payloads_with_angle_bkt_pattern = 0

# Paranthesis pattern
paranthesis_regex_pattern = r"\(|\)|%28|%29"
total_payloads_with_paranthesis_pattern = 0

# Semicolon pattern
semicolon_regex_pattern = r";|%3b"
total_payloads_with_semicolon_pattern = 0

# Slash pattern
slash_regex_pattern = r"/|%2f"
total_payloads_with_slash_pattern = 0

# Comment pattern
comment_regex_pattern = r"--|/\*|\*/|%2d%2d"
total_payloads_with_comment_pattern = 0

# Percent encoded pattern
percent_encoded_regex_pattern = r"%[0-9a-fA-F]{2}"
total_payloads_with_percent_encoded_pattern = 0

# SQL patterns
sql_patterns = [ "select", "union", "drop", "insert", "delete"]
total_payloads_per_sql_pattern = defaultdict(int)
for pattern in sql_patterns:
    total_payloads_per_sql_pattern[pattern] = 0

# Script Pattern
script_regex_pattern = r"<script|%3cscript"
total_payloads_with_script_pattern = 0

# Alert Pattern
alert_regex_pattern = r"alert|%61%6c%65%72%74"
total_payloads_with_alert_pattern = 0

# OR pattern
or_regex_pattern = r"\bor\b|\|\|"
total_payloads_with_or_pattern = 0

# Path traversal pattern
path_traversal_regex_pattern = r"\.\./|%2e%2e%2f"
total_payloads_with_path_traversal_pattern = 0


# Track label distribution for each pattern
payload_label_distribution = {
    "digits": {"anom": 0, "norm": 0},
    "parameter": {"anom": 0, "norm": 0},    
    "quote": {"anom": 0, "norm": 0},
    "angle_bkt": {"anom": 0, "norm": 0},
    "paranthesis": {"anom": 0, "norm": 0},
    "semicolon": {"anom": 0, "norm": 0},
    "slash": {"anom": 0, "norm": 0},
    "comment": {"anom": 0, "norm": 0},
    "percent_encoded": {"anom": 0, "norm": 0},
    
    "select": {"anom": 0, "norm": 0},
    "union": {"anom": 0, "norm": 0},
    "drop": {"anom": 0, "norm": 0},
    "insert": {"anom": 0, "norm": 0},
    "delete": {"anom": 0, "norm": 0},
 
    "script": {"anom": 0, "norm": 0},
    "alert": {"anom": 0, "norm": 0},
    "or": {"anom": 0, "norm": 0},
    "path_traversal": {"anom": 0, "norm": 0},    
}

# Iterate through each Payload and its corresponding label
for payload, label in zip(payload_df["payload"], payload_df["label"]):

    # Count special characters
    for char in special_chars:
        count = payload.count(char)
        if count > 0:
            total_payloads_per_special_char[char] += 1
        else:
            total_payloads_per_special_char[char] += 0

    if re.search(digits_regex_pattern, payload):
        total_payloads_with_digits_pattern += 1
        payload_label_distribution["digits"][label] += 1

    if re.search(parameter_regex_pattern, payload):
        total_payloads_with_parameter_pattern += 1
        payload_label_distribution["parameter"][label] += 1

    if re.search(quote_regex_pattern, payload):
        total_payloads_with_quote_pattern += 1
        payload_label_distribution["quote"][label] += 1
    
    if re.search(angle_bkt_regex_pattern, payload):
        total_payloads_with_angle_bkt_pattern += 1
        payload_label_distribution["angle_bkt"][label] += 1

    if re.search(paranthesis_regex_pattern, payload):
        total_payloads_with_paranthesis_pattern += 1
        payload_label_distribution["paranthesis"][label] += 1

    if re.search(semicolon_regex_pattern, payload):
        total_payloads_with_semicolon_pattern += 1
        payload_label_distribution["semicolon"][label] += 1
    
    if re.search(slash_regex_pattern, payload):
        total_payloads_with_slash_pattern += 1
        payload_label_distribution["slash"][label] += 1    

    if re.search(percent_encoded_regex_pattern, payload):
        total_payloads_with_percent_encoded_pattern += 1
        payload_label_distribution["percent_encoded"][label] += 1

    for pattern in sql_patterns:
        if pattern in payload:
          total_payloads_per_sql_pattern[pattern] +=  1
          payload_label_distribution[pattern][label] += 1
        
    if re.search(script_regex_pattern, payload):
        total_payloads_with_script_pattern += 1
        payload_label_distribution["script"][label] += 1
        
    if re.search(alert_regex_pattern, payload):
        total_payloads_with_alert_pattern += 1
        payload_label_distribution["alert"][label] += 1

    if re.search(or_regex_pattern, payload):
        total_payloads_with_or_pattern += 1
        payload_label_distribution["or"][label] += 1
        

    if re.search(path_traversal_regex_pattern, payload):
        total_payloads_with_path_traversal_pattern += 1
        payload_label_distribution["path_traversal"][label] += 1

# Display the total count of payloads containing each special character
print("Total payloads containing per special Character:")
for char, count in total_payloads_per_special_char.items():
    print(f"    {char}: {count} payloads")

print("Total payloads containing per SQL pattern:")
for pattern, count in total_payloads_per_sql_pattern.items():
    print(f"    {pattern}: {count} payloads")

print(f"Total payloads containing other patterns:")
print(f"   'parameter': {total_payloads_with_parameter_pattern}")
print(f"   'digits': {total_payloads_with_digits_pattern}")
print(f"   'quote': {total_payloads_with_quote_pattern}")
print(f"   'angle_bkt': {total_payloads_with_angle_bkt_pattern}")
print(f"   'paranthesis': {total_payloads_with_paranthesis_pattern}")
print(f"   'semicolon': {total_payloads_with_semicolon_pattern}")
print(f"   'slash': {total_payloads_with_slash_pattern}")
print(f"   'comment': {total_payloads_with_comment_pattern}")
print(f"   'percent_encoded': {total_payloads_with_percent_encoded_pattern}")
print(f"   'script': {total_payloads_with_script_pattern}")
print(f"   'alert': {total_payloads_with_alert_pattern}")
print(f"   'or': {total_payloads_with_or_pattern}")
print(f"   'path traversal': {total_payloads_with_path_traversal_pattern}")


# Display label distribution for each pattern
print("\nLabel distribution for patterns:")
for pattern, counts in payload_label_distribution.items():
    print(f"   {pattern}: {counts}")


Total payloads containing per special Character:
    &: 0 payloads
    =: 35574 payloads
    %: 21395 payloads
Total payloads containing per SQL pattern:
    select: 0 payloads
    union: 0 payloads
    drop: 4 payloads
    insert: 4588 payloads
    delete: 0 payloads
Total payloads containing other patterns:
   'parameter': 35574
   'digits': 32629
   'quote': 4188
   'angle_bkt': 0
   'paranthesis': 1554
   'semicolon': 0
   'slash': 17
   'comment': 0
   'percent_encoded': 21395
   'script': 0
   'alert': 1158
   'or': 2
   'path traversal': 0

Label distribution for patterns:
   digits: {'anom': 18629, 'norm': 14000}
   parameter: {'anom': 19574, 'norm': 16000}
   quote: {'anom': 4088, 'norm': 100}
   angle_bkt: {'anom': 0, 'norm': 0}
   paranthesis: {'anom': 1524, 'norm': 30}
   semicolon: {'anom': 0, 'norm': 0}
   slash: {'anom': 17, 'norm': 0}
   comment: {'anom': 0, 'norm': 0}
   percent_encoded: {'anom': 15001, 'norm': 6394}
   select: {'anom': 0, 'norm': 0}
   union: {'anom':

**Observations:** 
- Special Characters:
  - `&` no occurances and `=` is too common. Both these are not useful
  - `%` is also common but may be useful when combined with `percent_encoded` pattern
  > **Conclusion:**  Exclude `&` and `=`. Use `%` as a feature

- SQL Patterns:  
  -  No occurances `select` `union`, `delete`
  -  `insert` appears in both `anom` and `norm` but is more common in `anom` (56% of occurrences are anomalous).
  - `drop` appears only in `norm`, so it is not useful for anomaly detection.  
  > **Conclusion:** Include `insert` as binary feature. Exclude all other SQL patterns.  

- Other Patterns:
  - `>90%` of `quote` `paranthesis` `slash` `alert` are anomalous. Thsee are strong candidates features  
  - `>50%` of `Parameter` `percent_eccoded` `insert` are anomalous. Provides resonable discriminative value
  - Rest all are give almost no signals for anomalous classification 
  > **Conclusion:** Invlude `quote` `paranthesis` `slash` `alert` `Parameter` `percent_eccoded` `insert` drop the rest

Add the `payload` derived feature columns in the dataset

In [27]:
# Extract the `Payload` columns series
payload_series = df_featurised["payload"].astype(str)

# Count based features
df_featurised["payload_quote_count"] = payload_series.str.count(r"'|%27|\"|%22", flags=re.IGNORECASE)
df_featurised["payload_paren_count"] = payload_series.str.count(r"\(|\)|%28|%29", flags=re.IGNORECASE)
df_featurised["payload_slash_count"] = payload_series.str.count(r"/|%2f", flags=re.IGNORECASE)
df_featurised["payload_parameter_count"] = payload_series.str.count(r"(^|&)[^=&]+=", flags=re.IGNORECASE)
df_featurised["payload_percent_count"] = payload_series.str.count(r"%", flags=re.IGNORECASE)
df_featurised["payload_percent_encoded_count"] = payload_series.str.count(r"%[0-9a-fA-F]{2}", flags=re.IGNORECASE)

# Binary flag based features
df_featurised["payload_has_alert"] = payload_series.str.contains(r"alert|%61%6c%65%72%74", case=False, regex=True)
df_featurised["payload_has_sql_insert"] = payload_series.str.contains(r"insert", case=False, regex=True)

# Additional features
df_featurised["payload_length"] = payload_series.str.len()

df_featurised.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   url                            61065 non-null  object 
 1   host                           61065 non-null  object 
 2   label                          61065 non-null  object 
 3   payload                        61065 non-null  object 
 4   contentLength                  61065 non-null  float64
 5   method_GET                     61065 non-null  float64
 6   method_POST                    61065 non-null  float64
 7   method_PUT                     61065 non-null  float64
 8   host_localhost:8080            61065 non-null  float64
 9   host_localhost:9090            61065 non-null  float64
 10  url_has_select                 61065 non-null  int64  
 11  url_has_drop                   61065 non-null  int64  
 12  url_has_numeric_id             61065 non-null 

In [28]:
df_featurised.head(5)

,url,host,label,payload,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,host_localhost:9090,...,url_length,payload_quote_count,payload_paren_count,payload_slash_count,payload_parameter_count,payload_percent_count,payload_percent_encoded_count,payload_has_alert,payload_has_sql_insert,payload_length
0,http://localhost:8080/tienda1/imagenes/2.gif,localhost:8080,norm,,0.0,1.0,0.0,0.0,1.0,0.0,...,44,0,0,0,0,0,0,False,False,0
1,http://localhost:8080/tienda1/publico/vaciar.j...,localhost:8080,anom,,0.0,1.0,0.0,0.0,1.0,0.0,...,70,0,0,0,0,0,0,False,False,0
2,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,norm,id=2,0.0,1.0,0.0,0.0,1.0,0.0,...,57,0,0,0,1,0,0,False,False,4
3,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,anom,id=3 nombre=Jam%F3n+Ib%E9rico precio=%2500 can...,0.0,1.0,0.0,0.0,1.0,0.0,...,48,0,0,0,1,4,4,False,False,77
4,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,anom,id=2sessionid%3D12312312%26+username%3D%3Cscri...,174.0,0.0,1.0,0.0,1.0,0.0,...,57,2,0,5,1,18,18,False,False,174


### 3.6 Analyse `contentLength` column values

In [29]:
print(f"Unique values: {df_featurised['contentLength'].nunique()}")
df_featurised['contentLength'].describe()

Unique values: 383


count    61065.000000
mean        31.936903
std         75.660779
min          0.000000
25%          0.000000
50%          0.000000
75%         17.000000
max        836.000000
Name: contentLength, dtype: float64

**Observations:**  

- `50%` of requests have a contentLength of `0`
- mean `31.94` > median `0.0`, indicating most requests have small or no payloads, but a few have very large payloads.

Inspect possibility of anomaly pattern based on `contentLength` statistical range

In [30]:
content_length_df = df_featurised[["contentLength", "label"]]

In [31]:
# label distribution for each statistical range
content_length_label_distribution = {
    "0_to_10": {"anom": 0, "norm": 0},
    "10_to_50": {"anom": 0, "norm": 0},
    "50_to_200": {"anom": 0, "norm": 0},
    "greater_than_200": {"anom": 0, "norm": 0},
}
for contentLength, label in zip(content_length_df["contentLength"], content_length_df["label"]):
    # Categorize contentLength into bins based on the statistics
    if contentLength > 0 and contentLength <= 10:
        content_length_label_distribution["0_to_10"][label] += 1
    elif contentLength > 10 and contentLength <= 50:
        content_length_label_distribution["10_to_50"][label] += 1
    elif contentLength > 50 and contentLength <= 200:
        content_length_label_distribution["50_to_200"][label] += 1
    else: # >200
        content_length_label_distribution["greater_than_200"][label] += 1

# Display the label distribution for each statistical range
print("Label Distribution for contentLength Ranges:")
for range_key, label_distr in content_length_label_distribution.items():
    print(f"   {range_key}: {label_distr}")

Label Distribution for contentLength Ranges:
   0_to_10: {'anom': 683, 'norm': 1000}
   10_to_50: {'anom': 2469, 'norm': 3000}
   50_to_200: {'anom': 3894, 'norm': 2000}
   greater_than_200: {'anom': 18019, 'norm': 30000}


**Observation:**  
- The ratio of anomaly to normal does not monotonically increase or decrease with contentLength.
- Small payloads (0-10 bytes) and very large payloads (>200 bytes) have lower anomalies percentages (~40%).
- Medium payloads (50-200 bytes) have relatively higher anomaly percentage (~66%).

Inspect possibility of anomaly pattern based on contentLength statistical thresholds

In [32]:
import numpy as np
from collections import defaultdict

# Assuming content_length_df contains 'contentLength' and 'label' columns
content_length_df = df_featurised[["contentLength", "label"]]

# Define the statistical thresholds
stats = {
    "min": 0.0,
    "25%": 0.0,
    "50%": 0.0,
    "75%": 17.0,
    "max": 836.0,
    "mean": 31.936903,
    "std": 75.660779
}

# Initialize a dictionary to store label distribution for each statistical range
label_distribution = {
    "min_to_25%": {"anom": 0, "norm": 0},
    "25%_to_50%": {"anom": 0, "norm": 0},
    "50%_to_75%": {"anom": 0, "norm": 0},
    "75%_to_mean": {"anom": 0, "norm": 0},
    "mean_to_mean+std": {"anom": 0, "norm": 0},
    "mean+std_to_max": {"anom": 0, "norm": 0},
    "zero": {"anom": 0, "norm": 0}
}

# Iterate through each contentLength and its corresponding label
for contentLength, label in zip(content_length_df["contentLength"], content_length_df["label"]):
    # Categorize contentLength into bins based on the statistics
    if contentLength == 0:
        label_distribution["zero"][label] += 1
    elif 0 < contentLength <= stats["25%"]:
        label_distribution["min_to_25%"][label] += 1
    elif stats["25%"] < contentLength <= stats["50%"]:
        label_distribution["25%_to_50%"][label] += 1
    elif stats["50%"] < contentLength <= stats["75%"]:
        label_distribution["50%_to_75%"][label] += 1
    elif stats["75%"] < contentLength <= stats["mean"]:
        label_distribution["75%_to_mean"][label] += 1
    elif stats["mean"] < contentLength <= stats["mean"] + stats["std"]:
        label_distribution["mean_to_mean+std"][label] += 1
    elif contentLength > stats["mean"] + stats["std"]:
        label_distribution["mean+std_to_max"][label] += 1

# Display the label distribution for each statistical range
print("Label Distribution for contentLength Ranges:")
for range_name, counts in label_distribution.items():
    total = counts["anom"] + counts["norm"]
    anom_percent = (counts["anom"] / total * 100) if total > 0 else 0
    norm_percent = (counts["norm"] / total * 100) if total > 0 else 0
    print(f"   {range_name}: {counts} (anom: {anom_percent:.2f}%, norm: {norm_percent:.2f}%)")

Label Distribution for contentLength Ranges:
   min_to_25%: {'anom': 0, 'norm': 0} (anom: 0.00%, norm: 0.00%)
   25%_to_50%: {'anom': 0, 'norm': 0} (anom: 0.00%, norm: 0.00%)
   50%_to_75%: {'anom': 808, 'norm': 2000} (anom: 28.77%, norm: 71.23%)
   75%_to_mean: {'anom': 638, 'norm': 0} (anom: 100.00%, norm: 0.00%)
   mean_to_mean+std: {'anom': 4946, 'norm': 4000} (anom: 55.29%, norm: 44.71%)
   mean+std_to_max: {'anom': 3585, 'norm': 2000} (anom: 64.19%, norm: 35.81%)
   zero: {'anom': 15088, 'norm': 28000} (anom: 35.02%, norm: 64.98%)


**Observations:**  
- 75%_to_mean (17-32 bytes): 100% of payloads in this range are anomalous. However the absolute number of requests `638` are very less compared to the total.
- mean+std_to_max (>108 bytes): 64.19% of payloads in this range are anomalous.
- mean_to_mean+std: 55.29% of payloads in this range are anomalous.
- Other ranges are lesser anomalous percentages

Based on the analysis conducted on the `contentLength` feature, no distinct or consistent pattern emerged that could reliably distinguish between anomalous and normal payloads. For the current iteration, proceeding with the extracted features`is_17_to_32_bytes` and `is_empty_payload` in addition to the original `contentLength` column which already present in the dataset.

In [33]:
# is_17_to_32_bytes
df_featurised["is_17_to_32_bytes"] = (
    (df_featurised["contentLength"] >= 17) &
    (df_featurised["contentLength"] <= 32)
).astype(int)

# is_empty_payload
df_featurised["is_empty_payload"] = (
    df_featurised["contentLength"] == 0
).astype(int)

# Verify the new features
df_featurised.head()

,url,host,label,payload,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,host_localhost:9090,...,payload_paren_count,payload_slash_count,payload_parameter_count,payload_percent_count,payload_percent_encoded_count,payload_has_alert,payload_has_sql_insert,payload_length,is_17_to_32_bytes,is_empty_payload
0,http://localhost:8080/tienda1/imagenes/2.gif,localhost:8080,norm,,0.0,1.0,0.0,0.0,1.0,0.0,...,0,0,0,0,0,False,False,0,0,1
1,http://localhost:8080/tienda1/publico/vaciar.j...,localhost:8080,anom,,0.0,1.0,0.0,0.0,1.0,0.0,...,0,0,0,0,0,False,False,0,0,1
2,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,norm,id=2,0.0,1.0,0.0,0.0,1.0,0.0,...,0,0,1,0,0,False,False,4,0,1
3,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,anom,id=3 nombre=Jam%F3n+Ib%E9rico precio=%2500 can...,0.0,1.0,0.0,0.0,1.0,0.0,...,0,0,1,4,4,False,False,77,0,1
4,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,anom,id=2sessionid%3D12312312%26+username%3D%3Cscri...,174.0,0.0,1.0,0.0,1.0,0.0,...,0,5,1,18,18,False,False,174,0,0


In [38]:
print(df_featurised.shape)
print(df_featurised.columns.tolist())

(61065, 25)
['url', 'host', 'label', 'payload', 'contentLength', 'method_GET', 'method_POST', 'method_PUT', 'host_localhost:8080', 'host_localhost:9090', 'url_has_select', 'url_has_drop', 'url_has_numeric_id', 'url_length', 'payload_quote_count', 'payload_paren_count', 'payload_slash_count', 'payload_parameter_count', 'payload_percent_count', 'payload_percent_encoded_count', 'payload_has_alert', 'payload_has_sql_insert', 'payload_length', 'is_17_to_32_bytes', 'is_empty_payload']


### 3.7 Analyse `label` column values

In [39]:
df_featurised['label'].value_counts()

label
norm    36000
anom    25065
Name: count, dtype: int64

Create a binary target column `anomalous` with `1` for `anom` and `0` for `norm`

In [40]:
df_featurised["anomalous"] = (df_featurised["label"] == "anom").astype(int)
print(df_featurised.shape)
print(df_featurised.columns.tolist())

(61065, 26)
['url', 'host', 'label', 'payload', 'contentLength', 'method_GET', 'method_POST', 'method_PUT', 'host_localhost:8080', 'host_localhost:9090', 'url_has_select', 'url_has_drop', 'url_has_numeric_id', 'url_length', 'payload_quote_count', 'payload_paren_count', 'payload_slash_count', 'payload_parameter_count', 'payload_percent_count', 'payload_percent_encoded_count', 'payload_has_alert', 'payload_has_sql_insert', 'payload_length', 'is_17_to_32_bytes', 'is_empty_payload', 'anomalous']


## 3.8 Save the Feature-Engineered DataFrame to disk

Save the final DataFrame (with features) as `csic_2010_engineered.csv` for modeling.

In [41]:
df_featurised.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 26 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   url                            61065 non-null  object 
 1   host                           61065 non-null  object 
 2   label                          61065 non-null  object 
 3   payload                        61065 non-null  object 
 4   contentLength                  61065 non-null  float64
 5   method_GET                     61065 non-null  float64
 6   method_POST                    61065 non-null  float64
 7   method_PUT                     61065 non-null  float64
 8   host_localhost:8080            61065 non-null  float64
 9   host_localhost:9090            61065 non-null  float64
 10  url_has_select                 61065 non-null  int64  
 11  url_has_drop                   61065 non-null  int64  
 12  url_has_numeric_id             61065 non-null 

In [42]:
numeric_cols = df_featurised.select_dtypes(include=np.number).columns.tolist()
print(len(numeric_cols))
print(numeric_cols)
#categorical_cols = train_inputs.select_dtypes('object').columns.tolist()

20
['contentLength', 'method_GET', 'method_POST', 'method_PUT', 'host_localhost:8080', 'host_localhost:9090', 'url_has_select', 'url_has_drop', 'url_has_numeric_id', 'url_length', 'payload_quote_count', 'payload_paren_count', 'payload_slash_count', 'payload_parameter_count', 'payload_percent_count', 'payload_percent_encoded_count', 'payload_length', 'is_17_to_32_bytes', 'is_empty_payload', 'anomalous']


In [43]:
df_featurised[numeric_cols]

,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,host_localhost:9090,url_has_select,url_has_drop,url_has_numeric_id,url_length,payload_quote_count,payload_paren_count,payload_slash_count,payload_parameter_count,payload_percent_count,payload_percent_encoded_count,payload_length,is_17_to_32_bytes,is_empty_payload,anomalous
0,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,44,0,0,0,0,0,0,0,0,1,0
1,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,70,0,0,0,0,0,0,0,0,1,1
2,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,57,0,0,0,1,0,0,4,0,1,0
3,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,48,0,0,0,1,4,4,77,0,1,1
4,174.0,0.0,1.0,0.0,1.0,0.0,0,0,0,57,2,0,5,1,18,18,174,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61060,33.0,0.0,1.0,0.0,1.0,0.0,0,0,0,48,0,0,0,1,0,0,33,0,0,0
61061,248.0,0.0,1.0,0.0,1.0,0.0,0,0,0,49,0,0,0,1,3,3,248,0,0,1
61062,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,52,0,0,0,1,0,0,58,0,1,1
61063,38.0,0.0,1.0,0.0,1.0,0.0,0,0,0,47,0,0,0,1,0,0,38,0,0,0


In [45]:
%time
df_featurised[numeric_cols].to_parquet(output_path.with_suffix('.parquet'))

CPU times: user 12 μs, sys: 1 μs, total: 13 μs
Wall time: 30.5 μs
